# 24 - Comparación controlada del preprocesado con LoFTR en SB013

Este notebook evalúa si el preprocesado desarrollado para el TFM facilita la búsqueda de correspondencias entre la **H&E real** y la **HSI** del caso SB013.

A diferencia de la primera prueba exploratoria, todas las condiciones utilizan exactamente las mismas modalidades, la misma adquisición HSI `NRM_EDU`, la misma WSI H&E y la misma escala física intrapareja. La H&E se considera la imagen móvil y la HSI la imagen fija.

Se comparan tres condiciones:

1. **Escenas completas:** imágenes a escala común, sin aplicar las máscaras.
2. **Máscaras sin recorte:** se elimina el fondo, pero se conserva el campo de visión completo.
3. **Segmentadas y recortadas:** se aplican las máscaras y se recorta cada imagen al espécimen, como en el pipeline final.

LoFTR se emplea únicamente como herramienta de diagnóstico. Se utiliza el modelo preentrenado `outdoor` y no se realiza entrenamiento con los datos del proyecto.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import warnings
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import cv2
import kornia.feature as KF
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from PIL import Image

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 220
np.random.seed(42)
torch.manual_seed(42)
cv2.setRNGSeed(42)

ROOT = Path.cwd()
DL_ROOT = ROOT / "Registration" / "DeepLearning"
METHOD_ROOT = DL_ROOT / "Tecnicas_registration" / "05_feature_based_matching"
PREPROCESS_NOTEBOOK = DL_ROOT / "9_funcion_preprocesado.ipynb"
EXPORT_NOTEBOOK = DL_ROOT / "10_guardar_imagenes_a_escala.ipynb"
PREPARED_DIR = DL_ROOT / "Imagenes_a_escala" / "SB013"
OUTPUT_DIR = METHOD_ROOT / "outputs" / "outputs_loftr_SB013_preprocesado_controlado"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SIDE = 768
RANSAC_THRESHOLD_PX = 5.0
HIGH_CONFIDENCE_THRESHOLD = 0.5

print("Notebook:", Path.cwd())
print("Salidas:", OUTPUT_DIR)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Carga del pipeline validado

Se reutilizan las funciones del notebook 9 y las correcciones de segmentación validadas en el notebook 10. No se ejecutan las celdas de ejemplo ni el guardado global de los seis casos.


In [ ]:
def load_preprocessing_definitions():
    namespace = globals()

    notebook_9 = nbformat.read(PREPROCESS_NOTEBOOK, as_version=4)
    skipped = []
    for index, cell in enumerate(notebook_9.cells):
        if cell.cell_type != "code":
            continue
        if "EXAMPLE_ID =" in cell.source or "preprocess_output =" in cell.source:
            skipped.append(index)
            continue
        exec(compile(cell.source, f"notebook9_cell_{index}", "exec"), namespace)

    notebook_10 = nbformat.read(EXPORT_NOTEBOOK, as_version=4)
    override_cells = []
    for index, cell in enumerate(notebook_10.cells):
        if cell.cell_type == "code" and "def _largest_component_he_scale" in cell.source:
            exec(compile(cell.source, f"notebook10_cell_{index}", "exec"), namespace)
            override_cells.append(index)

    return {"notebook_9_skipped_cells": skipped, "notebook_10_override_cells": override_cells}


definition_info = load_preprocessing_definitions()
definition_info


## Preprocesado de SB013

La función principal se ejecuta sin visualizaciones intermedias. De su salida se recuperan las imágenes completas, las máscaras validadas, las escalas físicas y las imágenes finales segmentadas.


In [ ]:
SHOW_FLAGS = {
    "show_he_basica": False,
    "show_print_he_datos": False,
    "show_hsi_basica": False,
    "show_hsi_contorno_caja": False,
    "show_hsi_contorno_medidas": False,
    "show_lienzo_unico_caja": False,
    "show_he_segmentada": False,
    "show_hsi_segmentada": False,
    "show_lienzo_unico_segmentadas": False,
    "show_he_segmentada_escala": False,
    "show_hsi_segmentada_escala": False,
    "show_he_segmentada_escala_sin_barra": False,
    "show_hsi_segmentada_escala_sin_barra": False,
}

case_paths = SPECIMENS["SB013"]
preprocess_output = preprocesar_he_hsi(
    he_path=case_paths["he_path"],
    hsi_hdr_path=case_paths["hsi_hdr_path"],
    specimen_id="SB013",
    **SHOW_FLAGS,
)

scale_info = preprocess_output["same_scale_segmented_output"]["info"]
print("H&E original:", preprocess_output["he_rgb"].shape)
print("HSI original:", preprocess_output["hsi_adjusted_result"]["pseudo_rgb"].shape)
print("H&E final:", preprocess_output["he_segmentada_escala_rgb"].shape)
print("HSI final:", preprocess_output["hsi_segmentada_escala_rgb"].shape)
print(f"Escala física común: {scale_info['target_px_per_cm']:.4f} px/cm")


## Construcción de las condiciones controladas

Las imágenes completas se reescalan con los mismos factores físicos utilizados por el pipeline final. En cada condición se aplica un único factor de reducción a las dos modalidades para limitar el coste de LoFTR sin romper la escala común. Las máscaras no se entregan a LoFTR en la condición sin segmentación; únicamente se conservan para evaluar posteriormente la transformación estimada.


In [ ]:
def resize_mask(mask, width, height):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8) * 255
    return cv2.resize(mask_u8, (int(width), int(height)), interpolation=cv2.INTER_NEAREST) > 0


def apply_mask(image, mask):
    result = np.asarray(image, dtype=np.uint8).copy()
    result[~np.asarray(mask, dtype=bool)] = 0
    return result


def crop_to_mask(image, mask, padding=0):
    mask = np.asarray(mask, dtype=bool)
    ys, xs = np.where(mask)
    if len(xs) == 0:
        raise ValueError("La máscara está vacía")
    height, width = mask.shape
    x1 = max(0, int(xs.min()) - int(padding))
    y1 = max(0, int(ys.min()) - int(padding))
    x2 = min(width - 1, int(xs.max()) + int(padding))
    y2 = min(height - 1, int(ys.max()) + int(padding))
    return image[y1:y2 + 1, x1:x2 + 1], mask[y1:y2 + 1, x1:x2 + 1], (x1, y1, x2, y2)


def resize_pair_with_shared_factor(
    moving, fixed, moving_mask, fixed_mask, max_side=768, factor=None
):
    if factor is None:
        max_dimension = max(
            moving.shape[0], moving.shape[1], fixed.shape[0], fixed.shape[1]
        )
        factor = min(1.0, float(max_side) / float(max_dimension))
    factor = float(factor)

    def resize_one(image, mask):
        height, width = image.shape[:2]
        new_width = max(1, int(round(width * factor)))
        new_height = max(1, int(round(height * factor)))
        interpolation = cv2.INTER_AREA if factor < 1.0 else cv2.INTER_LINEAR
        image_resized = cv2.resize(image, (new_width, new_height), interpolation=interpolation)
        mask_resized = resize_mask(mask, new_width, new_height)
        return image_resized, mask_resized

    moving_resized, moving_mask_resized = resize_one(moving, moving_mask)
    fixed_resized, fixed_mask_resized = resize_one(fixed, fixed_mask)
    return moving_resized, fixed_resized, moving_mask_resized, fixed_mask_resized, factor


def pad_to_multiple(image, mask, multiple=8):
    height, width = image.shape[:2]
    padded_height = int(np.ceil(height / multiple) * multiple)
    padded_width = int(np.ceil(width / multiple) * multiple)
    pad_bottom = padded_height - height
    pad_right = padded_width - width
    image_padded = cv2.copyMakeBorder(
        image, 0, pad_bottom, 0, pad_right, cv2.BORDER_CONSTANT, value=(0, 0, 0)
    )
    mask_padded = cv2.copyMakeBorder(
        mask.astype(np.uint8), 0, pad_bottom, 0, pad_right, cv2.BORDER_CONSTANT, value=0
    ) > 0
    return image_padded, mask_padded


def prepare_condition(
    identifier, label, he_image, hsi_image, he_mask, hsi_mask, shared_factor
):
    moving, fixed, moving_mask, fixed_mask, factor = resize_pair_with_shared_factor(
        he_image,
        hsi_image,
        he_mask,
        hsi_mask,
        max_side=MAX_SIDE,
        factor=shared_factor,
    )
    moving, moving_mask = pad_to_multiple(moving, moving_mask)
    fixed, fixed_mask = pad_to_multiple(fixed, fixed_mask)
    return {
        "id": identifier,
        "label": label,
        "moving": moving,
        "fixed": fixed,
        "moving_mask": moving_mask,
        "fixed_mask": fixed_mask,
        "pair_resize_factor": factor,
        "effective_px_per_cm": float(scale_info["target_px_per_cm"] * factor),
    }


he_scale = scale_info["he_scale"]
hsi_scale = scale_info["hsi_scale"]
target_scale = float(scale_info["target_px_per_cm"])

he_full_scaled, he_full_resize = resize_to_target_px_per_cm(
    preprocess_output["he_rgb"],
    src_px_per_cm_x=he_scale["px_per_cm_x"],
    src_px_per_cm_y=he_scale["px_per_cm_y"],
    target_px_per_cm=target_scale,
)
hsi_full_scaled, hsi_full_resize = resize_to_target_px_per_cm(
    preprocess_output["hsi_adjusted_result"]["pseudo_rgb"],
    src_px_per_cm_x=hsi_scale["px_per_cm_x"],
    src_px_per_cm_y=hsi_scale["px_per_cm_y"],
    target_px_per_cm=target_scale,
)

he_full_mask = resize_mask(
    preprocess_output["he_tissue_mask"], he_full_scaled.shape[1], he_full_scaled.shape[0]
)
hsi_full_mask = resize_mask(
    preprocess_output["hsi_specimen_mask"], hsi_full_scaled.shape[1], hsi_full_scaled.shape[0]
)

he_masked_full = apply_mask(he_full_scaled, he_full_mask)
hsi_masked_full = apply_mask(hsi_full_scaled, hsi_full_mask)

# Se leen los artefactos finales para reproducir exactamente la entrada del pipeline.
he_final = np.asarray(Image.open(PREPARED_DIR / "SB013_h&e.png").convert("RGB"))
hsi_final = np.asarray(Image.open(PREPARED_DIR / "SB013_hsi.png").convert("RGB"))
he_final_mask = np.asarray(Image.open(PREPARED_DIR / "SB013_he_mask.png").convert("L")) > 0
hsi_final_mask = np.asarray(Image.open(PREPARED_DIR / "SB013_hsi_mask.png").convert("L")) > 0

global_max_dimension = max(
    he_full_scaled.shape[0],
    he_full_scaled.shape[1],
    hsi_full_scaled.shape[0],
    hsi_full_scaled.shape[1],
)
global_input_factor = min(1.0, float(MAX_SIDE) / float(global_max_dimension))
print(f"Factor global común para las tres condiciones: {global_input_factor:.6f}")

conditions = [
    prepare_condition(
        "unsegmented_full",
        "Escenas completas",
        he_full_scaled,
        hsi_full_scaled,
        he_full_mask,
        hsi_full_mask,
        global_input_factor,
    ),
    prepare_condition(
        "segmented_full",
        "Máscaras aplicadas sin recorte",
        he_masked_full,
        hsi_masked_full,
        he_full_mask,
        hsi_full_mask,
        global_input_factor,
    ),
    prepare_condition(
        "segmented_cropped",
        "Segmentadas y recortadas",
        he_final,
        hsi_final,
        he_final_mask,
        hsi_final_mask,
        global_input_factor,
    ),
]

condition_summary = pd.DataFrame(
    [
        {
            "condición": item["label"],
            "H&E móvil": f"{item['moving'].shape[1]}x{item['moving'].shape[0]}",
            "HSI fija": f"{item['fixed'].shape[1]}x{item['fixed'].shape[0]}",
            "factor de entrada": item["pair_resize_factor"],
            "px/cm efectivos": item["effective_px_per_cm"],
        }
        for item in conditions
    ]
)
display(condition_summary.round(3))


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 14), constrained_layout=True)
for row, condition in enumerate(conditions):
    axes[row, 0].imshow(condition["moving"])
    axes[row, 0].set_title(f"{condition['label']} | H&E móvil")
    axes[row, 1].imshow(condition["fixed"])
    axes[row, 1].set_title(f"{condition['label']} | HSI fija")
    for axis in axes[row]:
        axis.axis("off")
fig.suptitle("SB013: entradas controladas para LoFTR", fontsize=16)
fig.savefig(OUTPUT_DIR / "24_entradas_controladas_SB013.png", bbox_inches="tight")
plt.show()


## LoFTR y estimación de transformaciones

LoFTR obtiene correspondencias entre la H&E móvil y la HSI fija. A partir de ellas se estiman una transformación afín parcial y una homografía mediante RANSAC. Las máscaras se utilizan exclusivamente para medir el solapamiento después de aplicar cada transformación.


In [ ]:
def gray_tensor(image_rgb, device):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    return torch.from_numpy(gray)[None, None].to(device)


def run_loftr(matcher, moving, fixed, device):
    with torch.no_grad():
        correspondences = matcher(
            {
                "image0": gray_tensor(moving, device),
                "image1": gray_tensor(fixed, device),
            }
        )
    return {
        "points_moving": correspondences["keypoints0"].detach().cpu().numpy(),
        "points_fixed": correspondences["keypoints1"].detach().cpu().numpy(),
        "confidence": correspondences["confidence"].detach().cpu().numpy(),
    }


def estimate_transforms(points_moving, points_fixed):
    affine = affine_inliers = homography = homography_inliers = None
    if len(points_moving) >= 3:
        affine, affine_inliers = cv2.estimateAffinePartial2D(
            points_moving,
            points_fixed,
            method=cv2.RANSAC,
            ransacReprojThreshold=RANSAC_THRESHOLD_PX,
            maxIters=5000,
            confidence=0.995,
        )
    if len(points_moving) >= 4:
        homography, homography_inliers = cv2.findHomography(
            points_moving,
            points_fixed,
            method=cv2.RANSAC,
            ransacReprojThreshold=RANSAC_THRESHOLD_PX,
            maxIters=5000,
            confidence=0.995,
        )
    return {
        "affine": affine,
        "affine_inliers": affine_inliers,
        "homography": homography,
        "homography_inliers": homography_inliers,
    }


def warp_mask(mask, transform, target_shape, kind):
    height, width = target_shape
    source = mask.astype(np.uint8) * 255
    if transform is None:
        return np.zeros((height, width), dtype=bool)
    if kind == "affine":
        warped = cv2.warpAffine(source, transform, (width, height), flags=cv2.INTER_NEAREST)
    else:
        warped = cv2.warpPerspective(source, transform, (width, height), flags=cv2.INTER_NEAREST)
    return warped > 0


def warp_rgb(image, transform, target_shape, kind):
    height, width = target_shape
    if transform is None:
        return np.zeros((height, width, 3), dtype=np.uint8)
    if kind == "affine":
        return cv2.warpAffine(image, transform, (width, height), flags=cv2.INTER_LINEAR)
    return cv2.warpPerspective(image, transform, (width, height), flags=cv2.INTER_LINEAR)


def dice_score(mask_a, mask_b):
    a = np.asarray(mask_a, dtype=bool)
    b = np.asarray(mask_b, dtype=bool)
    denominator = int(a.sum()) + int(b.sum())
    return float(2 * np.logical_and(a, b).sum() / denominator) if denominator else 1.0


def iou_score(mask_a, mask_b):
    a = np.asarray(mask_a, dtype=bool)
    b = np.asarray(mask_b, dtype=bool)
    union = int(np.logical_or(a, b).sum())
    return float(np.logical_and(a, b).sum() / union) if union else 1.0


def count_inliers(inliers):
    return int(inliers.sum()) if inliers is not None else 0


def draw_matches(moving, fixed, points_moving, points_fixed, confidence, inliers=None, max_draw=70):
    height = max(moving.shape[0], fixed.shape[0])
    width = moving.shape[1] + fixed.shape[1]
    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    canvas[: moving.shape[0], : moving.shape[1]] = moving
    canvas[: fixed.shape[0], moving.shape[1] :] = fixed

    if len(points_moving) == 0:
        return canvas

    if inliers is not None and np.any(inliers):
        indices = np.flatnonzero(np.asarray(inliers).ravel() > 0)
        indices = indices[np.argsort(confidence[indices])[::-1]]
    else:
        indices = np.argsort(confidence)[::-1]
    indices = indices[:max_draw]

    colors = plt.cm.turbo(np.linspace(0.05, 0.95, max(1, len(indices))))[:, :3]
    colors = (colors * 255).astype(np.uint8)
    for color, index in zip(colors, indices):
        x0, y0 = points_moving[index]
        x1, y1 = points_fixed[index]
        rgb = tuple(int(value) for value in color)
        point_0 = (int(round(x0)), int(round(y0)))
        point_1 = (int(round(x1)) + moving.shape[1], int(round(y1)))
        cv2.circle(canvas, point_0, 3, rgb, -1, cv2.LINE_AA)
        cv2.circle(canvas, point_1, 3, rgb, -1, cv2.LINE_AA)
        cv2.line(canvas, point_0, point_1, rgb, 1, cv2.LINE_AA)
    return canvas


def make_overlay(fixed, fixed_mask, moving_warped, moving_mask_warped):
    overlay = cv2.addWeighted(fixed, 0.60, moving_warped, 0.40, 0)
    fixed_contours, _ = cv2.findContours(
        fixed_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
    )
    moving_contours, _ = cv2.findContours(
        moving_mask_warped.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
    )
    cv2.drawContours(overlay, fixed_contours, -1, (0, 255, 255), 2)
    cv2.drawContours(overlay, moving_contours, -1, (255, 0, 255), 2)
    return overlay


## Ejecución y métricas

Se registran el número total de correspondencias, las correspondencias con confianza igual o superior a 0,5, los inliers de RANSAC y las métricas Dice e IoU obtenidas al transformar la máscara H&E sobre la máscara HSI.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
matcher = KF.LoFTR(pretrained="outdoor").to(device).eval()
print("Dispositivo LoFTR:", device)

results = []
artifacts = {}

for condition in conditions:
    print(f"\n=== {condition['label']} ===")
    matches = run_loftr(matcher, condition["moving"], condition["fixed"], device)
    transforms = estimate_transforms(matches["points_moving"], matches["points_fixed"])
    target_shape = condition["fixed_mask"].shape

    affine_mask = warp_mask(condition["moving_mask"], transforms["affine"], target_shape, "affine")
    homography_mask = warp_mask(
        condition["moving_mask"], transforms["homography"], target_shape, "homography"
    )
    affine_rgb = warp_rgb(condition["moving"], transforms["affine"], target_shape, "affine")
    homography_rgb = warp_rgb(
        condition["moving"], transforms["homography"], target_shape, "homography"
    )

    affine_inliers = count_inliers(transforms["affine_inliers"])
    homography_inliers = count_inliers(transforms["homography_inliers"])
    num_matches = len(matches["points_moving"])

    row = {
        "condition_id": condition["id"],
        "condition": condition["label"],
        "num_matches": int(num_matches),
        "mean_confidence": float(np.mean(matches["confidence"])) if num_matches else 0.0,
        "median_confidence": float(np.median(matches["confidence"])) if num_matches else 0.0,
        "high_confidence_matches_ge_0_5": int(
            np.sum(matches["confidence"] >= HIGH_CONFIDENCE_THRESHOLD)
        ),
        "affine_inliers": affine_inliers,
        "affine_inlier_ratio": float(affine_inliers / num_matches) if num_matches else 0.0,
        "affine_dice": dice_score(affine_mask, condition["fixed_mask"]),
        "affine_iou": iou_score(affine_mask, condition["fixed_mask"]),
        "homography_inliers": homography_inliers,
        "homography_inlier_ratio": float(homography_inliers / num_matches) if num_matches else 0.0,
        "homography_dice": dice_score(homography_mask, condition["fixed_mask"]),
        "homography_iou": iou_score(homography_mask, condition["fixed_mask"]),
        "pair_resize_factor": float(condition["pair_resize_factor"]),
        "effective_px_per_cm": float(condition["effective_px_per_cm"]),
        "moving_shape": list(condition["moving"].shape),
        "fixed_shape": list(condition["fixed"].shape),
    }
    results.append(row)

    preferred_inliers = transforms["homography_inliers"]
    match_image = draw_matches(
        condition["moving"],
        condition["fixed"],
        matches["points_moving"],
        matches["points_fixed"],
        matches["confidence"],
        preferred_inliers,
    )
    affine_overlay = make_overlay(
        condition["fixed"], condition["fixed_mask"], affine_rgb, affine_mask
    )
    homography_overlay = make_overlay(
        condition["fixed"], condition["fixed_mask"], homography_rgb, homography_mask
    )

    Image.fromarray(match_image).save(OUTPUT_DIR / f"{condition['id']}_loftr_matches.png")
    Image.fromarray(affine_overlay).save(OUTPUT_DIR / f"{condition['id']}_affine_overlay.png")
    Image.fromarray(homography_overlay).save(
        OUTPUT_DIR / f"{condition['id']}_homography_overlay.png"
    )
    artifacts[condition["id"]] = {
        "matches": match_image,
        "affine_overlay": affine_overlay,
        "homography_overlay": homography_overlay,
    }

    print(
        f"matches={num_matches}, affine={affine_inliers} inliers, "
        f"homography={homography_inliers} inliers, "
        f"Dice homografía={row['homography_dice']:.3f}"
    )

metrics = pd.DataFrame(results)
metrics.to_csv(OUTPUT_DIR / "24_metricas_loftr_preprocesado_SB013.csv", index=False)
(OUTPUT_DIR / "24_metricas_loftr_preprocesado_SB013.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8"
)

display(
    metrics[
        [
            "condition",
            "num_matches",
            "high_confidence_matches_ge_0_5",
            "affine_inliers",
            "affine_inlier_ratio",
            "affine_dice",
            "homography_inliers",
            "homography_inlier_ratio",
            "homography_dice",
        ]
    ].round(3)
)


## Figuras de resultados

La primera figura resume las métricas de las tres condiciones. La segunda está preparada para la memoria y compara directamente las escenas completas con los especímenes segmentados y recortados.


In [ ]:
labels = metrics["condition"].tolist()
x = np.arange(len(labels))

fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)

axes[0].bar(x - 0.18, metrics["num_matches"], 0.36, label="Totales", color="#4C78A8")
axes[0].bar(
    x + 0.18,
    metrics["high_confidence_matches_ge_0_5"],
    0.36,
    label="Confianza ≥ 0,5",
    color="#F58518",
)
axes[0].set_title("Correspondencias LoFTR")
axes[0].set_ylabel("Número de correspondencias")
axes[0].legend()

axes[1].bar(x - 0.18, metrics["affine_inlier_ratio"], 0.36, label="Afín", color="#54A24B")
axes[1].bar(
    x + 0.18,
    metrics["homography_inlier_ratio"],
    0.36,
    label="Homografía",
    color="#E45756",
)
axes[1].set_title("Proporción de inliers RANSAC")
axes[1].set_ylim(0, 1)
axes[1].legend()

axes[2].bar(x - 0.18, metrics["affine_dice"], 0.36, label="Afín", color="#72B7B2")
axes[2].bar(
    x + 0.18,
    metrics["homography_dice"],
    0.36,
    label="Homografía",
    color="#B279A2",
)
axes[2].set_title("Dice después de la transformación")
axes[2].set_ylim(0, 1)
axes[2].legend()

short_labels = ["Completas", "Máscaras", "Segmentadas\ny recortadas"]
for axis in axes:
    axis.set_xticks(x)
    axis.set_xticklabels(short_labels)
    axis.grid(axis="y", alpha=0.25)

fig.suptitle("SB013: efecto del preprocesado sobre LoFTR", fontsize=15)
summary_path = OUTPUT_DIR / "24_resumen_metricas_loftr_SB013.png"
fig.savefig(summary_path, bbox_inches="tight")
plt.show()


In [ ]:
def pair_canvas(moving, fixed, gap=20):
    target_height = max(moving.shape[0], fixed.shape[0])

    def pad_height(image):
        difference = target_height - image.shape[0]
        top = difference // 2
        bottom = difference - top
        return cv2.copyMakeBorder(
            image, top, bottom, 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255)
        )

    separator = np.full((target_height, gap, 3), 255, dtype=np.uint8)
    return np.concatenate([pad_height(moving), separator, pad_height(fixed)], axis=1)


raw_condition = conditions[0]
masked_condition = conditions[1]
raw_inputs = pair_canvas(raw_condition["moving"], raw_condition["fixed"])
masked_inputs = pair_canvas(masked_condition["moving"], masked_condition["fixed"])

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
axes[0, 0].imshow(raw_inputs)
axes[0, 0].set_title("Entradas completas: H&E móvil | HSI fija")
axes[0, 1].imshow(artifacts["unsegmented_full"]["matches"])
axes[0, 1].set_title("Inliers LoFTR con escenas completas")
axes[1, 0].imshow(masked_inputs)
axes[1, 0].set_title("Entradas con las máscaras aplicadas")
axes[1, 1].imshow(artifacts["segmented_full"]["matches"])
axes[1, 1].set_title("Inliers LoFTR después de eliminar el fondo")
for axis in axes.ravel():
    axis.axis("off")
fig.suptitle("Caso SB013: comparación controlada del efecto del preprocesado", fontsize=16)
memory_figure_path = OUTPUT_DIR / "24_figura_memoria_comparacion_loftr_SB013.png"
fig.savefig(memory_figure_path, bbox_inches="tight")
plt.show()


In [ ]:
metrics_by_id = metrics.set_index("condition_id")
raw_metrics = metrics_by_id.loc["unsegmented_full"]
masked_metrics = metrics_by_id.loc["segmented_full"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
axes[0, 0].imshow(artifacts["unsegmented_full"]["matches"])
axes[0, 0].set_title(
    f"Escenas completas: {int(raw_metrics['num_matches'])} matches, "
    f"{int(raw_metrics['homography_inliers'])} inliers"
)
axes[0, 1].imshow(artifacts["unsegmented_full"]["homography_overlay"])
axes[0, 1].set_title(
    f"Escenas completas | Homografía: Dice = {raw_metrics['homography_dice']:.3f}"
)
axes[1, 0].imshow(artifacts["segmented_full"]["matches"])
axes[1, 0].set_title(
    f"Máscaras aplicadas: {int(masked_metrics['num_matches'])} matches, "
    f"{int(masked_metrics['homography_inliers'])} inliers"
)
axes[1, 1].imshow(artifacts["segmented_full"]["homography_overlay"])
axes[1, 1].set_title(
    f"Máscaras aplicadas | Homografía: Dice = {masked_metrics['homography_dice']:.3f}"
)
for axis in axes.ravel():
    axis.axis("off")
fig.suptitle(
    "Caso SB013: efecto de eliminar el fondo en el registro con LoFTR",
    fontsize=16,
)
results_memory_figure_path = (
    OUTPUT_DIR / "24_figura_memoria_resultados_loftr_SB013.png"
)
fig.savefig(results_memory_figure_path, bbox_inches="tight")
plt.show()

## Conclusiones automáticas

Las conclusiones siguientes se generan a partir de las métricas guardadas. Deben interpretarse como una comprobación exploratoria de un único caso, no como una validación general del método.


In [ ]:
by_id = {row["condition_id"]: row for row in results}
raw = by_id["unsegmented_full"]
masked = by_id["segmented_full"]
final = by_id["segmented_cropped"]

mask_match_delta = masked["num_matches"] - raw["num_matches"]
mask_homography_dice_delta = masked["homography_dice"] - raw["homography_dice"]
mask_affine_dice_delta = masked["affine_dice"] - raw["affine_dice"]
crop_match_delta = final["num_matches"] - masked["num_matches"]
crop_homography_dice_delta = final["homography_dice"] - masked["homography_dice"]
crop_affine_dice_delta = final["affine_dice"] - masked["affine_dice"]

conclusions = (
    "### Lectura de los resultados\n\n"
    f"- Las escenas completas producen **{raw['num_matches']} correspondencias**. "
    f"Al aplicar las máscaras sin recortar se obtienen **{masked['num_matches']}** "
    f"({mask_match_delta:+d}).\n"
    f"- La eliminación del fondo eleva el Dice afín de **{raw['affine_dice']:.3f}** "
    f"a **{masked['affine_dice']:.3f}** ({mask_affine_dice_delta:+.3f}) y el Dice de "
    f"homografía de **{raw['homography_dice']:.3f}** a "
    f"**{masked['homography_dice']:.3f}** ({mask_homography_dice_delta:+.3f}).\n"
    f"- El recorte independiente aumenta las correspondencias de "
    f"**{masked['num_matches']}** a **{final['num_matches']}** ({crop_match_delta:+d}), "
    f"pero reduce el Dice afín en {crop_affine_dice_delta:+.3f} y el Dice de homografía "
    f"en {crop_homography_dice_delta:+.3f}.\n"
    "- Por tanto, la evidencia favorable se concentra en la aplicación de las máscaras: "
    "eliminar el fondo facilita LoFTR, mientras que un recorte independiente no garantiza "
    "una transformación geométricamente mejor.\n"
    "- Un aumento del número de correspondencias no implica necesariamente una mejora "
    "geométrica: deben considerarse conjuntamente los inliers de RANSAC y el "
    "solapamiento de las máscaras.\n"
    "- El experimento utiliza únicamente SB013 y un LoFTR preentrenado sobre imágenes "
    "naturales. Sus resultados sirven como comprobación preliminar del preprocesado, "
    "pero no permiten generalizar al conjunto completo.\n"
)

(OUTPUT_DIR / "24_conclusiones_loftr_SB013.md").write_text(conclusions.strip() + "\n", encoding="utf-8")
display(Markdown(conclusions))

print("\nArchivos principales para la memoria:")
print(" -", results_memory_figure_path)
print(" -", memory_figure_path)
print(" -", summary_path)
print(" -", OUTPUT_DIR / "24_metricas_loftr_preprocesado_SB013.csv")
